<center><h2>Практическое задание</h2></center>


1. Реализовать кастомный класс свертки, используя только Numpy

2. Обучить CNN в PyTorch с использовании разработанного вами класса

<h2>Реализуем кастомный класс свертки

In [17]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader

In [ ]:
layer = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=12, stride=2, padding=2, dilation=3)
layer

Conv2d(1, 4, kernel_size=(12, 12), stride=(2, 2), padding=(2, 2), dilation=(3, 3))

Должны реализовать вот это...

За работу

In [ ]:
# import numpy as np
# import torch
# import torch.nn as nn
# from torch.autograd import Function

# # ==========================================
# # ЧАСТЬ 1: Оптимизированное ядро на NumPy
# # ==========================================

# def get_im2col_indices(x_shape, field_height, field_width, padding=1, stride=1):
#     """
#     Вычисляет индексы для преобразования im2col.
#     Это вспомогательная функция, чтобы векторизовать извлечение окон.
#     """
#     N, C, H, W = x_shape
#     out_height = (H + 2 * padding - field_height) // stride + 1
#     out_width = (W + 2 * padding - field_width) // stride + 1

#     i0 = np.repeat(np.arange(field_height), field_width)
#     i0 = np.tile(i0, C)
#     i1 = stride * np.repeat(np.arange(out_height), out_width)
#     j0 = np.tile(np.arange(field_width), field_height * C)
#     j1 = stride * np.tile(np.arange(out_width), out_height)
    
#     i = i0.reshape(-1, 1) + i1.reshape(1, -1)
#     j = j0.reshape(-1, 1) + j1.reshape(1, -1)
#     k = np.repeat(np.arange(C), field_height * field_width).reshape(-1, 1)
#     return (k, i, j)

# def im2col_indices(x, field_height, field_width, padding=1, stride=1):
#     """
#     Преобразует 4D тензор изображения в 2D матрицу столбцов.
#     """
#     p = padding
#     x_padded = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p)), mode='constant')
#     k, i, j = get_im2col_indices(x.shape, field_height, field_width, padding, stride)
    
#     # Магия NumPy: выборка элементов по индексам (векторизованно)
#     cols = x_padded[:, k, i, j]
#     C = x.shape[1]
    
#     # Reshape для матричного умножения: (C * Kh * Kw, N * Oh * Ow)
#     cols = cols.transpose(1, 2, 0).reshape(field_height * field_width * C, -1)
#     return cols

# def col2im_indices(cols, x_shape, field_height, field_width, padding=1, stride=1):
#     """
#     Обратная операция для Backward Pass. 
#     Складывает градиенты из колонок обратно в формат изображения.
#     """
#     N, C, H, W = x_shape
#     H_padded, W_padded = H + 2 * padding, W + 2 * padding
#     x_padded = np.zeros((N, C, H_padded, W_padded), dtype=cols.dtype)
    
#     k, i, j = get_im2col_indices(x_shape, field_height, field_width, padding, stride)
#     cols_reshaped = cols.reshape(C * field_height * field_width, -1, N)
#     cols_reshaped = cols_reshaped.transpose(2, 0, 1)
    
#     # np.add.at позволяет корректно суммировать перекрывающиеся градиенты
#     np.add.at(x_padded, (slice(None), k, i, j), cols_reshaped)
    
#     if padding == 0:
#         return x_padded
#     return x_padded[:, :, padding:-padding, padding:-padding]

# # ==========================================
# # ЧАСТЬ 2: PyTorch Autograd Function
# # ==========================================

# class NumpyConv2dFunction(Function):
#     @staticmethod
#     def forward(ctx, input, weight, bias=None, stride=1, padding=0):
#         # 1. Конвертируем входные тензоры в NumPy
#         # detach() нужен, чтобы оторвать от графа, cpu() - чтобы перенести в RAM
#         x = input.detach().cpu().numpy()
#         w = weight.detach().cpu().numpy()
        
#         # Параметры свертки
#         n_filters, d_filter, h_filter, w_filter = w.shape
#         n_x, d_x, h_x, w_x = x.shape
#         h_out = (h_x - h_filter + 2 * padding) // stride + 1
#         w_out = (w_x - w_filter + 2 * padding) // stride + 1
        
#         # 2. Основная операция (Im2Col + GEMM)
#         # Превращаем входную картинку в матрицу
#         x_col = im2col_indices(x, h_filter, w_filter, padding=padding, stride=stride)
        
#         # Превращаем веса фильтров в строку: (Num_Filters, C*Kh*Kw)
#         w_col = w.reshape(n_filters, -1)
        
#         # Матричное умножение (Самая тяжелая часть, делается через BLAS)
#         out = w_col @ x_col 
        
#         # Добавляем bias, если есть
#         if bias is not None:
#             b = bias.detach().cpu().numpy()
#             out += b.reshape(-1, 1)
        
#         # Решейп обратно в 4D тензор: (N, Out_Channels, H_out, W_out)
#         out = out.reshape(n_filters, h_out, w_out, n_x)
#         out = out.transpose(3, 0, 1, 2)
        
#         # 3. Сохраняем контекст для Backward Pass
#         ctx.save_for_backward(input, weight, bias)
#         ctx.stride = stride
#         ctx.padding = padding
#         ctx.x_col = x_col # Сохраняем развернутую матрицу входа (нужна для градиента весов)
        
#         # Возвращаем результат как PyTorch Tensor
#         return torch.from_numpy(out).float().to(input.device)

#     @staticmethod
#     def backward(ctx, grad_output):
#         # Достаем сохраненное
#         input, weight, bias = ctx.saved_tensors
#         stride, padding = ctx.stride, ctx.padding
#         x_col = ctx.x_col
        
#         # Градиент приходит с верхнего слоя. Конвертируем в NumPy.
#         grad_output = grad_output.detach().cpu().numpy()
        
#         n_filters, d_filter, h_filter, w_filter = weight.shape
        
#         # Подготовка grad_output для матричных операций
#         # (N, C_out, H_out, W_out) -> (C_out, N * H_out * W_out)
#         grad_output_reshaped = grad_output.transpose(1, 2, 3, 0).reshape(n_filters, -1)
        
#         # --- Градиент по Весам (dW) ---
#         # dW = grad_output @ input_cols.T
#         grad_weight = grad_output_reshaped @ x_col.T
#         grad_weight = grad_weight.reshape(weight.shape)
        
#         # --- Градиент по Bias (db) ---
#         # db = sum(grad_output)
#         if bias is not None:
#             grad_bias = np.sum(grad_output, axis=(0, 2, 3))
#         else:
#             grad_bias = None
            
#         # --- Градиент по Входу (dX) ---
#         # 1. Сначала прокидываем градиент через веса (умножаем на веса)
#         # dX_col = W.T @ grad_output
#         w_reshape = weight.detach().cpu().numpy().reshape(n_filters, -1)
#         grad_x_col = w_reshape.T @ grad_output_reshaped
        
#         # 2. Выполняем col2im (обратное преобразование столбцов в картинку)
#         grad_input = col2im_indices(grad_x_col, input.shape, h_filter, w_filter, padding, stride)
        
#         # Возвращаем градиенты как тензоры
#         # Порядок возврата должен соответствовать аргументам forward
#         return (torch.from_numpy(grad_input).float().to(input.device), 
#                 torch.from_numpy(grad_weight).float().to(input.device), 
#                 torch.from_numpy(grad_bias).float().to(input.device) if bias is not None else None,
#                 None, None) # stride и padding не требуют градиентов

# # ==========================================
# # ЧАСТЬ 3: Обертка nn.Module
# # ==========================================

# class MyConv2d(nn.Module):
#     def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
#         super().__init__()
#         self.stride = stride
#         self.padding = padding
#         self.kernel_size = kernel_size
#         self.in_channels = in_channels
#         self.out_channels = out_channels
        
#         # Инициализируем веса как nn.Parameter (чтобы оптимизатор их видел)
#         # Размерность: (Out, In, K, K)
#         k = kernel_size
#         self.weight = nn.Parameter(torch.Tensor(out_channels, in_channels, k, k))
#         self.bias = nn.Parameter(torch.Tensor(out_channels))
        
#         # Стандартная инициализация (Kaiming / Xavier можно добавить, тут простая uniform)
#         nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
#         fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
#         bound = 1 / np.sqrt(fan_in)
#         nn.init.uniform_(self.bias, -bound, bound)
        
#     def forward(self, x):
#         return NumpyConv2dFunction.apply(x, self.weight, self.bias, self.stride, self.padding)

Для нормальной работы мы не должны использовать циклы for, только матричных умножения. Следовательно, для этого необходимо получать индексы, с которыми мы будем работать, и передавать их и в перед и назад (для обучения)

In [117]:
import numpy as np
from torch.autograd import Function

In [143]:
def get_indices(image_shape, kernel_height, kernel_width, stride=1, padding=0):
    number_filters, channales, height, width = image_shape

    height_out = (height + 2 * padding - kernel_height) // stride + 1
    width_out = (width + 2 * padding - kernel_width) // stride + 1

    level1 = np.repeat(np.arange(kernel_height), kernel_width)
    level1 = np.tile(level1, channales)
    every_levels = stride * np.repeat(np.arange(height_out), width_out)
    i = level1.reshape(-1, 1) + every_levels.reshape(1, -1)

    slide1 = np.tile(np.arange(kernel_width), kernel_height)
    slide1 = np.tile(slide1, channales)
    every_slides = stride * np.tile(np.arange(width_out), height_out)
    j = slide1.reshape(-1, 1) + every_slides.reshape(1, -1)
    
    d = np.repeat(np.arange(channales), kernel_height * kernel_width).reshape(-1, 1)

    return (i, j, d)

def im2col(image, kernel_height, kernel_width, stride=1, padding=0):
    image_padded = np.pad(image, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')

    i, j, d = get_indices(image.shape, kernel_height, kernel_width, stride, padding)

    cols = image_padded[:, d, i, j]

    cols = cols.transpose(1, 2, 0).reshape(kernel_height * kernel_width * image.shape[1], -1)

    return cols

def col2im(cols, img_shape, kernel_height, kernel_width, stride=1, padding=0):
    N, C, H, W = img_shape
    H_padded, W_padded = H + 2 * padding, W + 2 * padding
    img_padded = np.zeros((N, C, H_padded, W_padded), dtype=cols.dtype)

    i, j, d = get_indices(img_shape, kernel_height, kernel_width, stride, padding)
    cols_reshaped = cols.reshape(C * kernel_height * kernel_width, -1, N)
    cols_reshaped = cols_reshaped.transpose(2, 0, 1)

    np.add.at(img_padded, (slice(None), d, i, j), cols_reshaped)

    if padding == 0:
        return img_padded
    return img_padded[:, :, padding:-padding, padding:-padding]


class NumpyConv2dFunction(Function):
    @staticmethod
    def forward(ctx, input, weight, bias=None, stride=1, padding=0):
        image = input.detach().cpu().numpy()
        w =  weight.detach().cpu().numpy()

        n_kernel, d_kernel, h_kernel, w_kernel = w.shape
        n_image, d_image, h_image, w_image = image.shape

        h_out = (h_image + 2 * padding - h_kernel) // stride + 1
        w_out = (w_image + 2 * padding - w_kernel) // stride + 1

        image_col = im2col(image, h_kernel, w_kernel, stride, padding)

        w_col = w.reshape(n_kernel, -1)

        out = w_col @ image_col

        if bias is not None:
            b = bias.detach().cpu().numpy()
            out += b.reshape(-1, 1)

        out = out.reshape(n_kernel, h_out, w_out, n_image)
        out = out.transpose(3, 0, 1, 2)

        ctx.save_for_backward(input, weight, bias)
        ctx.stride = stride
        ctx.padding = padding
        ctx.x_col = image_col

        return torch.from_numpy(out).float().to(input.device)

    @staticmethod
    def backward(ctx, grad_output):
        input, weight, bias = ctx.saved_tensors
        stride, padding = ctx.stride, ctx.padding
        x_col = ctx.x_col

        grad_output = grad_output.detach().cpu().numpy()

        n_kernels, d_kernels, h_kernel, w_kernel = weight.shape
        grad_output_reshaped = grad_output.transpose(1, 2, 3, 0).reshape(n_kernels, -1)

        grad_weight = grad_output_reshaped @ x_col.T
        grad_weight = grad_weight.reshape(weight.shape)

        if bias is not None:
            grad_bias = np.sum(grad_output, axis=(0, 2, 3))
        else:
            grad_bias = None

        w_reshape = weight.detach().cpu().numpy().reshape(n_kernels, -1)
        grad_x_col = w_reshape.T @ grad_output_reshaped

        grad_input = col2im(grad_x_col, input.shape, h_kernel, w_kernel, stride, padding)

        return (torch.from_numpy(grad_input).float().to(input.device),
                torch.from_numpy(grad_weight).float().to(input.device),
                torch.from_numpy(grad_bias).float().to(input.device) if bias is not None else None,
                None,
                None)


class MyConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride = 1, padding = 0):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        self.weight = nn.Parameter(torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.Tensor(out_channels))

        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / np.sqrt(fan_in)
        nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, image):
        return NumpyConv2dFunction.apply(image, self.weight, self.bias, self.stride, self.padding)


<h2>Обучим CNN с использованием своего класса

In [144]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = MyConv2d(in_channels=1, out_channels=4, kernel_size=2, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.conv2 = MyConv2d(in_channels=4, out_channels=8, kernel_size=3)
        self.pool2 = nn.MaxPool2d(kernel_size=3)
        self.conv3 = MyConv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2)
        self.linear1 = nn.Linear(in_features=64, out_features=32)
        self.linear2 = nn.Linear(in_features=32, out_features=10)
        self.activation = nn.ReLU()
        self.flatten = nn.Flatten()
        # Для преобразование выходов со сверточных слоев в единый массив для линейных слоев
        self.layers = nn.ModuleList([self.conv1, self.activation, self.pool1,
                       self.conv2, self.activation, self.pool2,
                       self.conv3, self.activation, self.pool3,
                       self.flatten, self.linear1, self.activation,
                       self.linear2])
        
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [145]:
train_data = datasets.MNIST('./MyCNN', train=True, download=True, transform=transforms.ToTensor())
valid_data = datasets.MNIST('./MyCNN', train=False, download=True, transform=transforms.ToTensor())


In [146]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=1, shuffle=False)

In [147]:
device = 'cpu'

In [148]:
model = CNN()
model.to(device)

CNN(
  (conv1): MyConv2d()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): MyConv2d()
  (pool2): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  (conv3): MyConv2d()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (linear1): Linear(in_features=64, out_features=32, bias=True)
  (linear2): Linear(in_features=32, out_features=10, bias=True)
  (activation): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layers): ModuleList(
    (0): MyConv2d()
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): MyConv2d()
    (4): ReLU()
    (5): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (6): MyConv2d()
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): Linear(in_features=64, out_features=32

In [149]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [150]:
import numpy as np

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss, valid_loss = [], []
    print('Starting epoch: ', epoch + 1)
    for data, targets in train_loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())

    model.eval()
    with torch.inference_mode():
        total_correct = 0
        total_samples = len(valid_loader.dataset)
        for data, targets in valid_loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            loss = criterion(outputs, targets)
            valid_loss.append(loss.item())
            classes = torch.argmax(outputs, dim=-1)
            correct = (classes == targets).sum().item()
            total_correct += correct
    print(f'epoch {epoch + 1} trains_loss: {np.mean(train_loss)} valid_loss: {np.mean(valid_loss)} accuracy: {100 * total_correct / total_samples}')

Starting epoch:  1
epoch 1 trains_loss: 1.6084461027463277 valid_loss: 0.6389272309066845 accuracy: 80.85
Starting epoch:  2
epoch 2 trains_loss: 0.4929431572198868 valid_loss: 0.3730929977551477 accuracy: 88.85
Starting epoch:  3
epoch 3 trains_loss: 0.3509561956048012 valid_loss: 0.2981012859448373 accuracy: 91.23
Starting epoch:  4


KeyboardInterrupt: 